# 11. Paoay: reference review and field checks

Review boundary-limited crop predictions, screening flags and original notes. Freeze the preliminary map and prepare separate January observation/correction records. No downloads, model fitting, automatic corrections or accuracy claims.


## 1. Setup

Finish 10 first. Optional CSV paths can point to completed records; defaults create editable templates once and preserve later edits. Original notebooks and historical reports remain archived.


In [ ]:
from pathlib import Path
import json
import sys
repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import review, field_review
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress
settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
calendar_file = None  # Optional CSV; leave None to use the preserved generated template.
map_pointer = paoay_root / "maps/map_stage.json"
review_root = paoay_root / "review"
review_pointer = review_root / "review_stage.json"
baseline_pointer = paoay_root / "crop_baseline/crop_baseline_stage.json"
screening_pointer = paoay_root / "screening/screening_stage.json"
field_root = paoay_root / "field_review"
field_pointer = field_root / "field_review_stage.json"
observation_file = None  # Optional completed observation CSV; otherwise use the preserved template.
correction_file = None  # Optional separate correction-proposal CSV.
for pointer in (review_pointer, field_pointer):
    write_stage_pointer(pointer, status="pending", reason="Notebook 11 reference/field review is incomplete")
stages = ["Verify matching map products", "Review boundaries and calendar context", "Prepare field records", "Save frozen review", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("11_paoay_reference_field_review", stages, report_folder.parent / "progress", {})


## 2. Verify matching products

Use the current successful context, crop-baseline and screening pointers together. Pending or mismatched inputs stop the review; an older successful report is not selected as fallback.


In [ ]:
pipeline_progress.begin_stage("Verify matching map products")
map_handoff = resolve_stage_pointer(map_pointer)
baseline_handoff = resolve_stage_pointer(baseline_pointer)
screening_handoff = resolve_stage_pointer(screening_pointer)
context_request = review.prepare_request(map_handoff, review_root, local_path(calendar_file) if calendar_file else None)
print("Optional calendar:", context_request["calendar_file"])
print("Historical notes are context, not dated ground truth.")
pipeline_progress.end_stage()


## 3. Review boundaries

Compare predictions with original notes and optional calendar interpretations. Rotations, unmodelled crops and mixed/seedling areas retain their caveats. Unknown is not other crops; flags are not hard exclusions.


In [ ]:
pipeline_progress.begin_stage("Review boundaries and calendar context")
def area_progress(items):
    return pipeline_progress.track(items, "Review reference areas", unit="area")
review_handoff = review.build(context_request, iterator=area_progress)
context_bundle = review.load_review_bundle(review_handoff)
show_table([{**r, "pixels": int(r["pixels"]), "area_ha": float(r["area_ha"]), "roi_percent": float(r["roi_percent"])}
            for r in context_bundle["crop_baseline"] if int(r["pixels"]) > 0], title="Crop-model predictions by boundary")
show_table([{key:r[key] for key in ("name","crop","scope","status","months_overlapping_window")} for r in context_bundle["calendar"]], title="Optional calendar context — not accuracy")
pipeline_progress.end_stage()


## 4. Prepare January records

Record date, GPS, actual crop/cover, growth stage, evidence/photo reference and observer before checking the prediction where practical. Blank templates are not observations. Use one ID per observation and link separate correction proposals to that ID. A point does not label a whole polygon. Farmer recall and historical records are separate evidence types. Dates are explicit; no visit year is assumed.

A visit records that day. To claim support for a four-month historical window, document that entire period explicitly; this remains a manual evidence claim. General KMZ calendars cannot provide it. See the [field-check guide](../../docs/paoay_field_checks.md).


In [ ]:
pipeline_progress.begin_stage("Prepare field records")
request = field_review.prepare_request(baseline_handoff, screening_handoff, review_handoff, field_root,
                                      local_path(observation_file) if observation_file else None,
                                      local_path(correction_file) if correction_file else None)
print("Editable observations:", request["observation_file"])
print("Separate correction proposals:", request["correction_file"])
print("Reference suitability:", request["suitability_file"])
print("Templates contain no invented observations or crop labels.")
pipeline_progress.end_stage()


## 5. Freeze and save

Create an immutable preliminary-map copy and snapshots of the evidence, calendar and suitability files. Rerunning after edits creates a new report; older snapshots remain readable. No corrected raster or validation score is generated.


In [ ]:
pipeline_progress.begin_stage("Save frozen review")
field_handoff = field_review.build(request)
bundle = field_review.load_bundle(field_handoff)
show_table([{**r, "pixels": int(r["pixels"]), "boundary_percent": float(r["boundary_percent"])}
            for r in bundle["screening"] if int(r["pixels"]) > 0], title="Screening flags by boundary — overlapping, not exclusions")
show_table(bundle["observations"], title="Recorded observations and temporal context")
write_stage_pointer(review_pointer, review_handoff)
write_stage_pointer(field_pointer, field_handoff)
print("Readable report:", bundle["folder"] / "report.md")
print("Frozen preliminary map:", bundle["folder"] / "preliminary_crop_map.tif")
print("Field checklist without predicted labels:", bundle["folder"] / "field_check_plan.csv")
pipeline_progress.end_stage()


## 6. Next action

The five-notebook workflow is complete. Use the separate editable records for field work, then rerun 11 to save a new review. Correction proposals do not alter predictions. A future corrected raster requires evidence and an explicit spatial scope; it must remain separate from the original model map. Other-crop training methods remain TODOs.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified field-review handoff:", field_handoff)
print("Current field-review pointer:", field_pointer)
print("Observations:", bundle["manifest"]["observation_count"])
print("Correction proposals:", bundle["manifest"]["correction_proposal_count"])
print("Independent local accuracy evaluated: no. Corrected map created: no.")
print("Model, imagery and original predictions preserved.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
